# Racial Composition of the Universities

In [55]:
#setup chunk
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from IPython.display import IFrame, Markdown, display
import statsmodels.formula.api as smf

We need to filter the data in the same way in which it is filtered in Glynn's analysis. 

In [56]:
#data cleaning process-- especially making smaller datasets that are more manageable and only include relevant variables

dat = pd.read_csv('../data/Glynn_data/combined_data.csv')
dat.columns

Index(['univ_id', 'univ_name', 'state', 'zip_code', 'percent_pell',
       'completion_rate_fry', 'ugds_white', 'ugds_black', 'ugds_hisp',
       'ugds_asian', 'ugds_aian', 'ugds_nhpi', 'ugds_2mor', 'ugds_nra',
       'ugds_unkn', 'ugds_men', 'ugds_women', 'md_earn_wne_4yr', 'year',
       'hbcu', 'pbi', 'aanapii', 'hsi', 'tribal', 'long', 'all_male',
       'all_female', 'admit_rate', 'lat', 'pct_w_ba_hometown', 'degree_type'],
      dtype='str')

In [57]:
ugds_cols = [colname for colname in dat.columns if colname.startswith('ugds_')]
ugds_cols

['ugds_white',
 'ugds_black',
 'ugds_hisp',
 'ugds_asian',
 'ugds_aian',
 'ugds_nhpi',
 'ugds_2mor',
 'ugds_nra',
 'ugds_unkn',
 'ugds_men',
 'ugds_women']

As part of the filtering, we need to exclude universities that are not part of the 50 US states and universities that are not primarily 4-year institutions

In [58]:
#then narrow only to universities in upper-50 US (no PR or Guam)
dat['state'].value_counts()
non_50 = ['VI', 'AS', 'MP', 'FM', 'PW', 'MH', 'GU', 'PR']
    #virgin islands, american samoa, northern marina islands, federated state of micronesia, palao, guam, puerto rico

dat[~dat['state'].isin(non_50)].nunique() #taking out states not in the 50 states gives us 51 unique states (since the dataset includes DC)
#filtering
dat = dat[~dat['state'].isin(non_50)]

In [60]:
#keep only 4-yr institutions
dat = dat[dat['degree_type']==3]

In [61]:
dat

,univ_id,univ_name,state,zip_code,percent_pell,completion_rate_fry,ugds_white,ugds_black,ugds_hisp,ugds_asian,...,aanapii,hsi,tribal,long,all_male,all_female,admit_rate,lat,pct_w_ba_hometown,degree_type
0,100654,Alabama A & M University,AL,35762,0.6536,0.2678,0.0198,0.8955,0.0110,0.0019,...,0.0,0.0,0.0,-86.568502,0.0,0.0,0.5795,34.783368,13,3
1,100663,University of Alabama at Birmingham,AL,35294-0110,0.3308,0.6442,0.5130,0.2528,0.0711,0.0819,...,0.0,0.0,0.0,-86.799345,0.0,0.0,0.8818,33.505697,15.9300003051757,3
2,100690,Amridge University,AL,36117-3553,0.7769,0.5000,0.2851,0.6623,0.0307,0.0000,...,0.0,0.0,0.0,-86.174010,0.0,0.0,NaN,32.362609,13.2299995422363,3
3,100706,University of Alabama in Huntsville,AL,35899,0.2173,0.6295,0.7102,0.0873,0.0666,0.0389,...,0.0,0.0,0.0,-86.640449,0.0,0.0,0.6857,34.724557,17.6700000762939,3
4,100724,Alabama State University,AL,36104-0271,0.6976,0.2773,0.0155,0.9251,0.0121,0.0015,...,0.0,0.0,0.0,-86.295677,0.0,0.0,0.9755,32.364317,11.8100004196167,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5720,498049,Arizona College of Nursing-Southfield,MI,48033-7157,0.7206,NaN,0.2121,0.6397,0.0808,0.0236,...,0.0,0.0,0.0,-83.260202,0.0,0.0,1.0000,42.481560,10.9499998092651,3
5735,498447,Arizona College of Nursing-Falls Church,VA,22042-4566,NaN,NaN,0.1597,0.5694,0.1528,0.0556,...,0.0,0.0,0.0,-77.218156,0.0,0.0,1.0000,38.860531,10.9499998092651,3
5736,498456,Arizona College of Nursing-Ontario,CA,91761-1201,NaN,NaN,0.1048,0.1333,0.5381,0.1000,...,0.0,0.0,0.0,-117.578071,0.0,0.0,1.0000,34.066760,10.9499998092651,3
5740,498562,Commonwealth University of Pennsylvania,PA,17815,0.3184,0.5356,0.8005,0.0627,0.0638,0.0117,...,0.0,0.0,0.0,-76.447844,0.0,0.0,0.9309,41.007820,13.2200002670288,3


In [62]:
dat = dat[['univ_id','univ_name','state','zip_code','ugds_white',
 'ugds_black',
 'ugds_hisp',
 'ugds_asian',
 'ugds_aian',
 'ugds_nhpi',
 'ugds_2mor',
 'ugds_unkn',
 'ugds_men',
 'ugds_women',
          'lat','long']]

In [63]:
race_cols = ['ugds_white', 'ugds_black', 'ugds_hisp', 'ugds_asian', 'ugds_aian', 
             'ugds_nhpi', 'ugds_2mor']


# mapping column names to labels
#      white, black, hispanic, asian, amarican indian and alaska native, 
#     native hawaiian and pacific islander, two or more races, norace, unknown, 
#     white nonhispanic, blacknonhispanic, asian pacific islander
#

col_labels = {
    'ugds_white': 'White_alone', 
    'ugds_black': 'Black_alone', 
    'ugds_hisp': 'Hispanic_Latino', 
    'ugds_asian': 'Asian_alone', 
    'ugds_aian': 'AIAN_alone', 
    'ugds_nhpi': 'NHPI_alone', 
    'ugds_2mor': 'Two_or_More',
    'ugds_unkn': "Other_alone",
     'ugds_men': "Men",
     'ugds_women': "Women"
}


In [65]:
dat = dat.rename(columns=col_labels)

Now that we have the data we need, we need to create the vectors that will be needed to combine the analysis with the data on census racial composition

In [66]:
dat

,univ_id,univ_name,state,zip_code,White_alone,Black_alone,Hispanic_Latino,Asian_alone,AIAN_alone,NHPI_alone,Two_or_More,Other_alone,Men,Women,lat,long
0,100654,Alabama A & M University,AL,35762,0.0198,0.8955,0.0110,0.0019,0.0025,0.0015,0.0127,0.0435,0.4055,0.5945,34.783368,-86.568502
1,100663,University of Alabama at Birmingham,AL,35294-0110,0.5130,0.2528,0.0711,0.0819,0.0016,0.0005,0.0491,0.0064,0.3752,0.6248,33.505697,-86.799345
2,100690,Amridge University,AL,36117-3553,0.2851,0.6623,0.0307,0.0000,0.0044,0.0044,0.0000,0.0132,0.3640,0.6360,32.362609,-86.174010
3,100706,University of Alabama in Huntsville,AL,35899,0.7102,0.0873,0.0666,0.0389,0.0087,0.0016,0.0465,0.0252,0.5981,0.4019,34.724557,-86.640449
4,100724,Alabama State University,AL,36104-0271,0.0155,0.9251,0.0121,0.0015,0.0021,0.0009,0.0118,0.0088,0.3595,0.6405,32.364317,-86.295677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5720,498049,Arizona College of Nursing-Southfield,MI,48033-7157,0.2121,0.6397,0.0808,0.0236,0.0034,0.0034,0.0370,0.0000,0.0673,0.9327,42.481560,-83.260202
5735,498447,Arizona College of Nursing-Falls Church,VA,22042-4566,0.1597,0.5694,0.1528,0.0556,0.0000,0.0347,0.0278,0.0000,0.0625,0.9375,38.860531,-77.218156
5736,498456,Arizona College of Nursing-Ontario,CA,91761-1201,0.1048,0.1333,0.5381,0.1000,0.0048,0.0143,0.1048,0.0000,0.1095,0.8905,34.066760,-117.578071
5740,498562,Commonwealth University of Pennsylvania,PA,17815,0.8005,0.0627,0.0638,0.0117,0.0037,0.0008,0.0254,0.0278,0.3914,0.6086,41.007820,-76.447844


In [74]:
zip_dic = pd.read_csv('../data/zip_county_092022.csv')

In [75]:
print(zip_dic.columns.tolist())

['ZIP', 'COUNTY', 'USPS_ZIP_PREF_CITY', 'USPS_ZIP_PREF_STATE', 'RES_RATIO', 'BUS_RATIO', 'OTH_RATIO', 'TOT_RATIO']


In [76]:
# 1. Sort the crosswalk so the highest ratio is at the top for each ZIP
zip_dic_sorted = zip_dic.sort_values(by=['ZIP', 'RES_RATIO'], ascending=[True, False])

# 2. Drop duplicates, keeping only the first (which is now the highest ratio)
zip_dic_primary = zip_dic_sorted.drop_duplicates(subset=['ZIP'], keep='first')

In [78]:
# 1. Force both ZIP columns to be 5-digit strings
# This prevents '02138' (string) failing to match 2138 (number)
dat['zip_code'] = dat['zip_code'].astype(str).str.split('-').str[0].str.split('.').str[0].str.zfill(5)

# Change 'ZIP' below to the actual column name in zip_dic
zip_dic_primary['ZIP'] = zip_dic_primary['ZIP'].astype(str).str.zfill(5)
# 2. Merge the data
# We keep all rows from 'dat' and bring in the county from 'zip_dic'
dat = dat.merge(zip_dic_primary[['ZIP', 'COUNTY']], 
                left_on='zip_code', 
                right_on='ZIP', 
                how='left')

# 3. Clean up
dat = dat.drop(columns=['ZIP'])

print("Success! Here is a sample:")
print(dat[['univ_name', 'zip_code', 'COUNTY']].head())

Success! Here is a sample:
                             univ_name zip_code  COUNTY
0             Alabama A & M University    35762  1089.0
1  University of Alabama at Birmingham    35294  1073.0
2                   Amridge University    36117  1101.0
3  University of Alabama in Huntsville    35899  1089.0
4             Alabama State University    36104  1101.0


In [79]:
# 1. Fill NaNs with a placeholder or handle them
# 2. Convert to integer to get rid of the '.0'
# 3. Convert to string and pad with 'zfill' to ensure it is 5 digits long
dat['COUNTY'] = dat['COUNTY'].fillna(0).astype(int).astype(str).str.zfill(5)

# If you want to keep missing values as empty instead of '00000':
import numpy as np
dat['COUNTY'] = dat['COUNTY'].replace('00000', np.nan)

print(dat[['univ_name', 'COUNTY']].head())

                             univ_name COUNTY
0             Alabama A & M University  01089
1  University of Alabama at Birmingham  01073
2                   Amridge University  01101
3  University of Alabama in Huntsville  01089
4             Alabama State University  01101


In [80]:
dat

,univ_id,univ_name,state,zip_code,White_alone,Black_alone,Hispanic_Latino,Asian_alone,AIAN_alone,NHPI_alone,Two_or_More,Other_alone,Men,Women,lat,long,COUNTY
0,100654,Alabama A & M University,AL,35762,0.0198,0.8955,0.0110,0.0019,0.0025,0.0015,0.0127,0.0435,0.4055,0.5945,34.783368,-86.568502,01089
1,100663,University of Alabama at Birmingham,AL,35294,0.5130,0.2528,0.0711,0.0819,0.0016,0.0005,0.0491,0.0064,0.3752,0.6248,33.505697,-86.799345,01073
2,100690,Amridge University,AL,36117,0.2851,0.6623,0.0307,0.0000,0.0044,0.0044,0.0000,0.0132,0.3640,0.6360,32.362609,-86.174010,01101
3,100706,University of Alabama in Huntsville,AL,35899,0.7102,0.0873,0.0666,0.0389,0.0087,0.0016,0.0465,0.0252,0.5981,0.4019,34.724557,-86.640449,01089
4,100724,Alabama State University,AL,36104,0.0155,0.9251,0.0121,0.0015,0.0021,0.0009,0.0118,0.0088,0.3595,0.6405,32.364317,-86.295677,01101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1893,498049,Arizona College of Nursing-Southfield,MI,48033,0.2121,0.6397,0.0808,0.0236,0.0034,0.0034,0.0370,0.0000,0.0673,0.9327,42.481560,-83.260202,26125
1894,498447,Arizona College of Nursing-Falls Church,VA,22042,0.1597,0.5694,0.1528,0.0556,0.0000,0.0347,0.0278,0.0000,0.0625,0.9375,38.860531,-77.218156,51059
1895,498456,Arizona College of Nursing-Ontario,CA,91761,0.1048,0.1333,0.5381,0.1000,0.0048,0.0143,0.1048,0.0000,0.1095,0.8905,34.066760,-117.578071,06071
1896,498562,Commonwealth University of Pennsylvania,PA,17815,0.8005,0.0627,0.0638,0.0117,0.0037,0.0008,0.0254,0.0278,0.3914,0.6086,41.007820,-76.447844,42037


In [81]:
race_columns = ['White_alone', 'Hispanic_Latino', 'Black_alone', 'AIAN_alone', 
                'Asian_alone', 'NHPI_alone', 'Other_alone', 'Two_or_More']


In [82]:
dat_proportions = dat[race_columns]

In [84]:
# 3. Create your new vector dataframe
dat_vectors = dat[['COUNTY', 'univ_id','univ_name']].copy()

# 4. Store these proportions as a vector
dat_vectors['race_proportion_vector'] = dat_proportions.values.tolist()

# Display the first few rows to see the decimal proportions
dat_vectors.head()

,COUNTY,univ_id,univ_name,race_proportion_vector
0,01089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015..."
1,01073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005..."
2,01101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ..."
3,01089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001..."
4,01101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000..."


In [85]:
# 1. Calculate the sum of squares for each row in the proportions dataframe
# We square the numbers first (** 2) and then sum them horizontally (axis=1)
dat_vectors['diversity_index'] = (dat_proportions ** 2).sum(axis=1).round(4)

# Display to see the new concentration metric
dat_vectors.head()

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index
0,01089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045
1,01073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413
2,01101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211
3,01089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208
4,01101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564


In [88]:
# Store the merged data to manipulate with the census data later
dat_vectors.to_csv('../data/racial_comp_univ.csv', index=False)

In [87]:
dat_vectors

,COUNTY,univ_id,univ_name,race_proportion_vector,diversity_index
0,01089,100654,Alabama A & M University,"[0.0198, 0.011, 0.8955, 0.0025, 0.0019, 0.0015...",0.8045
1,01073,100663,University of Alabama at Birmingham,"[0.513, 0.0711, 0.2528, 0.0016, 0.0819, 0.0005...",0.3413
2,01101,100690,Amridge University,"[0.2851, 0.0307, 0.6623, 0.0044, 0.0, 0.0044, ...",0.5211
3,01089,100706,University of Alabama in Huntsville,"[0.7102, 0.0666, 0.0873, 0.0087, 0.0389, 0.001...",0.5208
4,01101,100724,Alabama State University,"[0.0155, 0.0121, 0.9251, 0.0021, 0.0015, 0.000...",0.8564
...,...,...,...,...,...
1893,26125,498049,Arizona College of Nursing-Southfield,"[0.2121, 0.0808, 0.6397, 0.0034, 0.0236, 0.003...",0.4627
1894,51059,498447,Arizona College of Nursing-Falls Church,"[0.1597, 0.1528, 0.5694, 0.0, 0.0556, 0.0347, ...",0.3781
1895,06071,498456,Arizona College of Nursing-Ontario,"[0.1048, 0.5381, 0.1333, 0.0048, 0.1, 0.0143, ...",0.3395
1896,42037,498562,Commonwealth University of Pennsylvania,"[0.8005, 0.0638, 0.0627, 0.0037, 0.0117, 0.000...",0.6504
